# short_horizon_risk_cap_counterfactual_v1

## Goal

Diagnose whether the latest state-conditioned BC candidate is mainly blocked by missing short-horizon risk representation or by the actor/checkpoint itself.

This notebook runs a fixed-actor daily-path counterfactual: keep each official warm-start actor path unchanged, recompute fold-local risk masks from qlib features, and cap the already-produced position when a lagged short-horizon risk signal is active.

## Governance

- Baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `state_conditioned_multi_objective_bc_screen_v1` and `drawdown_interval_risk_state_coverage_v1`
- Role: `diagnose`
- Changed layer: `model_selection` / `execution` / `risk_state_representation`
- Target: official warm-start walk-forward windows, `relative_return >= 30pp` and `MaxDD <= 15%`
- Accepted trade-off: no promotion from this notebook; a passing counterfactual only authorizes an official replay/runtime integration screen
- Failure mode: cap reduces MaxDD by mechanically suppressing exposure but loses the `+30pp` excess target, or only works with same-day/non-feasible signals

## External references

- Qlib workflow and data handler pattern: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL as an external reference for portfolio RL/risk-control evaluation loops: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- DAgger / imitation-learning caution for behavior cloning drift: https://proceedings.mlr.press/v15/ross11a.html


In [1]:

from __future__ import annotations

import json
import math
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "short_horizon_risk_cap_counterfactual_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "daily_paths"
DAILY_DIR.mkdir(parents=True, exist_ok=True)
for stale_daily_path in DAILY_DIR.glob("*.csv"):
    stale_daily_path.unlink()
SOURCE_CARD = ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1"
RISK_CARD = ROOT / "runtime/model_selection/card/drawdown_interval_risk_state_coverage_v1"
REFERENCE_CARD = ROOT / "runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
CAP_LEVELS = [0.0, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
PRIMARY_TIMING = "lag1"
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
FEATURE_COLS = [
    "anchor_gap_180", "anchor_slope_180", "trend_gap_21", "trend_gap_63", "trend_slope_21", "trend_slope_63",
    "mom_21", "mom_63", "realized_vol_21", "realized_vol_63", "vol_of_vol_21", "drawdown_21", "drawdown_63",
    "cum_drop_30", "loss_streak_21", "new_low_ratio_21", "downside_semivol_21", "vix_percentile_252",
    "tracking_error_vol_21", "relative_strength_21", "regime_risk", "regime_risk_no_vix", "regime_risk_velocity_5",
]
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def safe_slug(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def gate_loss(relative_return: float, max_dd: float) -> float:
    return max(0.0, TARGET_RELATIVE_RETURN - float(relative_return)) + 10.0 * max(0.0, float(max_dd) - TARGET_MAX_DD)

source_summary = load_json(SOURCE_CARD / "summary.json")
source_selection = load_json(SOURCE_CARD / "selection.json")
risk_summary = load_json(RISK_CARD / "summary.json")
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)

print("source_decision", source_summary["decision"])
print("selected_source_candidate", source_summary["selected_candidate"])
print("risk_handoff", risk_summary["next_mechanism"])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[90374:MainThread](2026-05-26 19:04:35,115) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[90374:MainThread](2026-05-26 19:04:35,774) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[90374:MainThread](2026-05-26 19:04:35,774) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


source_decision state_conditioned_multi_objective_drawdown_failed_no_promotion
selected_source_candidate multi_obj_floor45_cap20_riskfirst
risk_handoff add short-horizon drawdown/volshock risk state and use drawdown-interval-aware checkpoint scoring before any new floor/PPO


## Design

This is deliberately not another PPO/training sweep. The test is:

1. Load the current official warm-start daily paths for the state-conditioned BC candidates.
2. Rebuild the same fold-local risk masks from qlib feature frames and the previous diagnostic thresholds.
3. Apply one-trading-day-lagged risk signals as a cap on the actor's already-emitted positions.
4. Recompute price-return NAV, relative return, MaxDD, interval diagnostics, and gate loss.
5. Treat any passing result as diagnostic only, because official replay and runtime execution integration are still separate layers.


In [2]:

def build_signal_masks(features: pd.DataFrame, thresholds: dict[str, float]) -> dict[str, pd.Series]:
    existing = (
        ((features["drawdown_63"] <= thresholds["drawdown_63_q20"]) & (features["trend_slope_63"] <= thresholds["trend_slope_63_q55"]))
        | ((features["vix_percentile_252"] >= thresholds["vix_percentile_252_q80"]) & (features["realized_vol_63"] >= thresholds["realized_vol_63_q80"]))
        | ((features["regime_risk"] >= thresholds["regime_risk_q80"]) & (features["mom_63"] <= thresholds["mom_63_q40"]))
        | ((features["regime_risk_no_vix"] >= thresholds["regime_risk_no_vix_q80"]) & (features["tracking_error_vol_21"] >= thresholds["tracking_error_vol_21_q80"]))
    )
    short_drawdown = (
        (features["drawdown_21"] <= thresholds["drawdown_21_q20"])
        | (features["cum_drop_30"] >= thresholds["cum_drop_30_q80"])
        | (features["loss_streak_21"] >= thresholds["loss_streak_21_q80"])
        | (features["new_low_ratio_21"] >= thresholds["new_low_ratio_21_q80"])
    )
    short_volshock = (
        (features["realized_vol_21"] >= thresholds["realized_vol_21_q80"])
        | (features["vol_of_vol_21"] >= thresholds["vol_of_vol_21_q80"])
        | (features["downside_semivol_21"] >= thresholds["downside_semivol_21_q80"])
    )
    trend_rollover = (features["trend_slope_21"] <= thresholds["trend_slope_21_q40"]) & (features["mom_63"] >= thresholds["mom_63_q60"])
    risk_regime = (
        (features["regime_risk"] >= thresholds["regime_risk_q80"])
        | (features["tracking_error_vol_21"] >= thresholds["tracking_error_vol_21_q80"])
        | (features["vix_percentile_252"] >= thresholds["vix_percentile_252_q80"])
    )
    two_of_four = (short_drawdown.astype(int) + short_volshock.astype(int) + trend_rollover.astype(int) + risk_regime.astype(int)) >= 2
    masks = {
        "existing_long_horizon_risk": existing.fillna(False),
        "short_drawdown_or_newlow": short_drawdown.fillna(False),
        "short_volshock": short_volshock.fillna(False),
        "trend_rollover_after_momentum": trend_rollover.fillna(False),
        "risk_regime_or_tracking_error": risk_regime.fillna(False),
        "short_drawdown_and_vol": (short_drawdown & short_volshock).fillna(False),
        "two_of_four_short_risk": two_of_four.fillna(False),
    }
    masks["existing_or_two_of_four_short_risk"] = (masks["existing_long_horizon_risk"] | masks["two_of_four_short_risk"]).fillna(False)
    masks["existing_or_short_volshock"] = (masks["existing_long_horizon_risk"] | masks["short_volshock"]).fillna(False)
    masks["existing_or_short_drawdown"] = (masks["existing_long_horizon_risk"] | masks["short_drawdown_or_newlow"]).fillna(False)
    return masks

threshold_table = pd.read_csv(RISK_CARD / "fold_local_signal_thresholds.csv")
threshold_by_fold = {row["fold"]: {k: float(v) for k, v in row.items() if k != "fold"} for row in threshold_table.to_dict("records")}
feature_cache: dict[str, pd.DataFrame] = {}
mask_cache: dict[str, pd.DataFrame] = {}

for fold, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    features = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").reset_index()[["datetime", *FEATURE_COLS]].copy()
    features["datetime"] = pd.to_datetime(features["datetime"]).dt.normalize()
    test = features[(features["datetime"] >= pd.Timestamp(spec["test"][0])) & (features["datetime"] <= pd.Timestamp(spec["test"][1]))].copy()
    masks = build_signal_masks(test, threshold_by_fold[fold])
    mask_frame = test[["datetime"]].copy()
    for name, mask in masks.items():
        mask_frame[name] = mask.astype(bool).to_numpy()
    feature_cache[fold] = test
    mask_cache[fold] = mask_frame

signal_coverage_rows = []
for fold, mask_frame in mask_cache.items():
    for col in [c for c in mask_frame.columns if c != "datetime"]:
        signal_coverage_rows.append({"fold": fold, "signal_family": col, "test_signal_share": float(mask_frame[col].mean()), "test_signal_days": int(mask_frame[col].sum())})
signal_coverage = pd.DataFrame(signal_coverage_rows)
signal_coverage.to_csv(CARD / "signal_test_coverage.csv", index=False)
display(signal_coverage.sort_values(["fold", "test_signal_share"], ascending=[True, False]).head(20))


[90374:MainThread](2026-05-26 19:04:38,060) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[90374:MainThread](2026-05-26 19:04:38,064) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[90374:MainThread](2026-05-26 19:04:38,068) INFO - qlib.timer - [log.py:127] - Time cost: 0.009s | Init data Done


[90374:MainThread](2026-05-26 19:04:40,678) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[90374:MainThread](2026-05-26 19:04:40,681) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[90374:MainThread](2026-05-26 19:04:40,683) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


,fold,signal_family,test_signal_share,test_signal_days
2,wf_2022,short_volshock,1.000000,251
8,wf_2022,existing_or_short_volshock,1.000000,251
7,wf_2022,existing_or_two_of_four_short_risk,0.928287,233
9,wf_2022,existing_or_short_drawdown,0.908367,228
6,wf_2022,two_of_four_short_risk,0.896414,225
0,wf_2022,existing_long_horizon_risk,0.828685,208
1,wf_2022,short_drawdown_or_newlow,0.808765,203
5,wf_2022,short_drawdown_and_vol,0.808765,203
4,wf_2022,risk_regime_or_tracking_error,0.701195,176
3,wf_2022,trend_rollover_after_momentum,0.027888,7


In [3]:

def derive_actions(position: pd.Series) -> pd.Series:
    pos = pd.to_numeric(position, errors="coerce").fillna(0.0)
    delta = pos.diff().fillna(0.0)
    return pd.Series(np.where(delta > 1e-9, "BUY", np.where(delta < -1e-9, "SELL", "HOLD")), index=position.index)


def replay_price_path(daily: pd.DataFrame, effective_position: pd.Series, *, label: str) -> pd.DataFrame:
    frame = daily.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"]).dt.normalize()
    frame = frame.sort_values("datetime").reset_index(drop=True)
    price = pd.to_numeric(frame["price"], errors="coerce").ffill().bfill().astype(float).to_numpy()
    pos = pd.to_numeric(effective_position.reset_index(drop=True), errors="coerce").fillna(0.0).clip(0.0, 1.0).to_numpy()
    nav = np.empty(len(frame), dtype=float)
    base_nav = float(pd.to_numeric(frame["strategy_nav"], errors="coerce").iloc[0])
    nav[0] = base_nav
    for idx in range(1, len(frame)):
        asset_ret = price[idx] / price[idx - 1] - 1.0 if price[idx - 1] else 0.0
        nav[idx] = nav[idx - 1] * (1.0 + pos[idx - 1] * asset_ret)
    out = frame.copy()
    out["source_strategy_nav"] = pd.to_numeric(frame["strategy_nav"], errors="coerce")
    out["base_position"] = pd.to_numeric(frame["position"], errors="coerce").fillna(0.0)
    out["position"] = pos
    out["strategy_nav"] = nav
    out["buyhold_nav"] = base_nav * price / price[0]
    out["excess"] = out["strategy_nav"] - out["buyhold_nav"]
    out["excess_pct"] = (out["strategy_nav"] / out["buyhold_nav"] - 1.0) * 100.0
    out["action"] = derive_actions(out["position"])
    out["replay_label"] = label
    return out


def apply_cap(daily: pd.DataFrame, signal: pd.Series, cap: float) -> tuple[pd.Series, pd.Series]:
    base_position = pd.to_numeric(daily["position"], errors="coerce").fillna(0.0).clip(0.0, 1.0).reset_index(drop=True)
    trigger = signal.reset_index(drop=True).astype(bool)
    effective = base_position.copy()
    effective.loc[trigger] = np.minimum(effective.loc[trigger], float(cap))
    return effective, trigger


def enrich_with_signal(daily: pd.DataFrame, fold: str, signal_family: str, timing: str) -> pd.Series:
    frame = daily.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"]).dt.normalize()
    mask = mask_cache[fold][["datetime", signal_family]].copy()
    merged = frame[["datetime"]].merge(mask, on="datetime", how="left")
    raw = merged[signal_family].fillna(False).astype(bool)
    if timing == "lag1":
        return raw.shift(1, fill_value=False).astype(bool)
    if timing == "same_day_diagnostic":
        return raw.astype(bool)
    raise ValueError(f"unknown timing: {timing}")


def path_row(candidate: str, fold: str, source_path: Path, replayed: pd.DataFrame, signal_family: str, timing: str, cap: float, trigger: pd.Series) -> dict[str, Any]:
    metrics = daily_path_metrics(replayed)
    interval = max_drawdown_interval_metrics(replayed)
    changed = np.abs(pd.to_numeric(replayed["position"], errors="coerce") - pd.to_numeric(replayed["base_position"], errors="coerce")) > 1e-9
    interval_mask = (pd.to_datetime(replayed["datetime"]) >= pd.Timestamp(interval["peak_date"])) & (pd.to_datetime(replayed["datetime"]) <= pd.Timestamp(interval["trough_date"])) if interval["peak_date"] else pd.Series(False, index=replayed.index)
    return {
        "candidate": candidate,
        "fold": fold,
        "signal_family": signal_family,
        "timing": timing,
        "cap": float(cap),
        "source_daily_path": str(source_path.relative_to(ROOT)),
        "rows": metrics["rows"],
        "start": metrics["start"],
        "end": metrics["end"],
        "return": metrics["return"],
        "buy_and_hold_return": metrics["buy_and_hold_return"],
        "relative_return": metrics["relative_return"],
        "max_dd": metrics["max_dd"],
        "avg_pos": metrics["avg_pos"],
        "full_gate": bool(metrics["relative_return"] >= TARGET_RELATIVE_RETURN and metrics["max_dd"] <= TARGET_MAX_DD),
        "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])),
        "changed_days": int(changed.sum()),
        "capped_days": int((changed & trigger.reset_index(drop=True)).sum()),
        "signal_days": int(trigger.sum()),
        "signal_share": float(trigger.mean()) if len(trigger) else 0.0,
        "position_delta_mean": float((pd.to_numeric(replayed["base_position"], errors="coerce") - pd.to_numeric(replayed["position"], errors="coerce")).mean()),
        "peak_date": interval["peak_date"],
        "trough_date": interval["trough_date"],
        "interval_days": interval["interval_days"],
        "interval_avg_pos": interval["interval_avg_pos"],
        "interval_max_pos": interval["interval_max_pos"],
        "interval_signal_share": float(trigger.reset_index(drop=True).loc[interval_mask.reset_index(drop=True)].mean()) if bool(interval_mask.any()) else 0.0,
    }

manifest = pd.read_csv(SOURCE_CARD / "daily_manifest.csv")
official_metrics = pd.read_csv(SOURCE_CARD / "warm_start_official_metrics.csv")

baseline_rows = []
reconstruction_rows = []
metric_rows = []
daily_outputs: dict[tuple[str, str, str, str, float], pd.DataFrame] = {}
signal_families = [
    "existing_long_horizon_risk",
    "short_drawdown_or_newlow",
    "short_volshock",
    "short_drawdown_and_vol",
    "two_of_four_short_risk",
    "existing_or_two_of_four_short_risk",
    "existing_or_short_volshock",
    "existing_or_short_drawdown",
]

for item in manifest.to_dict("records"):
    candidate = str(item["candidate"])
    fold = str(item["fold"])
    source_path = ROOT / str(item["path"])
    daily = pd.read_csv(source_path)
    official_row = official_metrics[(official_metrics["candidate"].eq(candidate)) & (official_metrics["fold"].eq(fold))].iloc[0].to_dict()
    official_daily_metrics = daily_path_metrics(daily)
    reconstructed = replay_price_path(daily, pd.to_numeric(daily["position"], errors="coerce"), label="price_replay_original_position")
    reconstructed_metrics = daily_path_metrics(reconstructed)
    baseline_rows.append({
        "candidate": candidate,
        "fold": fold,
        "source_daily_path": str(source_path.relative_to(ROOT)),
        "official_relative_return": official_row["official_warm_relative_return"],
        "official_max_dd": official_row["official_warm_max_dd"],
        "official_avg_pos": official_row["official_warm_avg_pos"],
        "price_replay_relative_return": reconstructed_metrics["relative_return"],
        "price_replay_max_dd": reconstructed_metrics["max_dd"],
        "price_replay_avg_pos": reconstructed_metrics["avg_pos"],
        "price_replay_gate_loss": gate_loss(float(reconstructed_metrics["relative_return"]), float(reconstructed_metrics["max_dd"])),
    })
    actual_nav = pd.to_numeric(daily["strategy_nav"], errors="coerce").reset_index(drop=True)
    replay_nav = pd.to_numeric(reconstructed["strategy_nav"], errors="coerce").reset_index(drop=True)
    reconstruction_rows.append({
        "candidate": candidate,
        "fold": fold,
        "source_daily_path": str(source_path.relative_to(ROOT)),
        "max_abs_nav_error_pct": float((replay_nav / actual_nav - 1.0).abs().max() * 100.0),
        "final_nav_error_pct": float((replay_nav.iloc[-1] / actual_nav.iloc[-1] - 1.0) * 100.0),
        "official_relative_return": official_daily_metrics["relative_return"],
        "price_replay_relative_return": reconstructed_metrics["relative_return"],
        "relative_return_delta_pp": float(reconstructed_metrics["relative_return"] - official_daily_metrics["relative_return"]),
        "official_max_dd": official_daily_metrics["max_dd"],
        "price_replay_max_dd": reconstructed_metrics["max_dd"],
        "max_dd_delta_pp": float(reconstructed_metrics["max_dd"] - official_daily_metrics["max_dd"]),
    })
    for signal_family in signal_families:
        for timing in ["lag1", "same_day_diagnostic"]:
            trigger = enrich_with_signal(daily, fold, signal_family, timing)
            for cap in CAP_LEVELS:
                effective, trigger_used = apply_cap(daily, trigger, cap)
                replayed = replay_price_path(daily, effective, label=f"{signal_family}_{timing}_cap{cap:.2f}")
                metric_rows.append(path_row(candidate, fold, source_path, replayed, signal_family, timing, cap, trigger_used))
                key = (candidate, fold, signal_family, timing, float(cap))
                daily_outputs[key] = replayed

baseline_metrics = pd.DataFrame(baseline_rows)
reconstruction_error = pd.DataFrame(reconstruction_rows)
counterfactual_metrics = pd.DataFrame(metric_rows)
baseline_metrics.to_csv(CARD / "baseline_price_replay_metrics.csv", index=False)
reconstruction_error.to_csv(CARD / "reconstruction_error.csv", index=False)
counterfactual_metrics.to_csv(CARD / "counterfactual_metrics.csv", index=False)

display(reconstruction_error.sort_values("max_abs_nav_error_pct", ascending=False))
display(counterfactual_metrics.sort_values("gate_loss").head(20))


,candidate,fold,source_daily_path,max_abs_nav_error_pct,final_nav_error_pct,official_relative_return,price_replay_relative_return,relative_return_delta_pp,official_max_dd,price_replay_max_dd,max_dd_delta_pp
3,multi_obj_floor55_cap25_balanced,wf_2023_2024,runtime/model_selection/card/state_conditioned...,14.361289,11.566591,8.057465,20.514780,12.457314,19.036155,15.619059,-3.417096
0,multi_obj_floor45_cap20_riskfirst,wf_2022,runtime/model_selection/card/state_conditioned...,6.718618,6.718618,59.388029,64.279180,4.891151,30.626799,30.784137,0.157338
5,multi_obj_floor65_cap20_weighted,wf_2023_2024,runtime/model_selection/card/state_conditioned...,4.351765,-3.317087,22.619594,18.564025,-4.055569,18.949796,19.239563,0.289767
4,multi_obj_floor65_cap20_weighted,wf_2022,runtime/model_selection/card/state_conditioned...,3.117081,-1.071073,56.421922,55.673951,-0.747972,31.567204,31.955687,0.388483
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,runtime/model_selection/card/state_conditioned...,2.278896,-0.411188,30.262521,29.728363,-0.534158,16.122029,14.775225,-1.346805
2,multi_obj_floor55_cap25_balanced,wf_2022,runtime/model_selection/card/state_conditioned...,2.248560,0.374851,55.825533,56.085071,0.259537,31.808525,31.597367,-0.211158


,candidate,fold,signal_family,timing,cap,source_daily_path,rows,start,end,return,...,capped_days,signal_days,signal_share,position_delta_mean,peak_date,trough_date,interval_days,interval_avg_pos,interval_max_pos,interval_signal_share
190,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,existing_or_two_of_four_short_risk,same_day_diagnostic,0.05,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,33.157599,...,110,138,0.549801,0.048079,2024-03-07,2024-04-19,31,0.154041,0.512402,0.548387
84,multi_obj_floor45_cap20_riskfirst,wf_2022,existing_or_short_volshock,lag1,0.00,runtime/model_selection/card/state_conditioned...,250,2022-01-03,2022-12-29,-0.142857,...,249,249,0.996000,0.164365,2022-01-03,2022-01-04,2,0.050000,0.100000,0.500000
167,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_drawdown_and_vol,same_day_diagnostic,0.20,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,30.530403,...,3,82,0.326693,0.001535,2024-03-07,2024-04-19,31,0.222731,0.512402,0.322581
595,multi_obj_floor65_cap20_weighted,wf_2023_2024,short_volshock,same_day_diagnostic,0.00,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,36.835833,...,111,118,0.470120,0.059287,2024-07-10,2024-11-15,92,0.032977,0.479761,0.869565
596,multi_obj_floor65_cap20_weighted,wf_2023_2024,short_volshock,same_day_diagnostic,0.05,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,36.762481,...,108,118,0.470120,0.037230,2024-07-10,2024-08-07,21,0.146391,0.479761,0.761905
168,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,two_of_four_short_risk,lag1,0.00,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,32.007716,...,111,129,0.513944,0.065190,2024-03-07,2024-04-04,20,0.210958,0.512402,0.300000
169,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,two_of_four_short_risk,lag1,0.05,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,33.956313,...,101,129,0.513944,0.043193,2024-03-07,2024-04-19,31,0.168131,0.512402,0.516129
170,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,two_of_four_short_risk,lag1,0.08,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,33.407405,...,76,129,0.513944,0.032975,2024-03-07,2024-04-19,31,0.183615,0.512402,0.516129
171,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,two_of_four_short_risk,lag1,0.10,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,33.208675,...,61,129,0.513944,0.027153,2024-03-07,2024-04-19,31,0.193807,0.512402,0.516129
172,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,two_of_four_short_risk,lag1,0.12,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,32.948164,...,55,129,0.513944,0.022727,2024-03-07,2024-04-19,31,0.201748,0.512402,0.516129


In [4]:

primary = counterfactual_metrics[counterfactual_metrics["timing"].eq(PRIMARY_TIMING)].copy()
agg = (
    primary.groupby(["candidate", "signal_family", "timing", "cap"], dropna=False)
    .agg(
        fold_count=("fold", "nunique"),
        full_gate_count=("full_gate", "sum"),
        gate_loss=("gate_loss", "sum"),
        max_max_dd=("max_dd", "max"),
        min_relative_return=("relative_return", "min"),
        mean_relative_return=("relative_return", "mean"),
        mean_avg_pos=("avg_pos", "mean"),
        total_changed_days=("changed_days", "sum"),
        total_capped_days=("capped_days", "sum"),
        total_signal_days=("signal_days", "sum"),
        worst_interval_avg_pos=("interval_avg_pos", "max"),
        mean_interval_signal_share=("interval_signal_share", "mean"),
    )
    .reset_index()
)
agg["full_gate_all"] = agg["full_gate_count"].eq(agg["fold_count"])
agg["relative_margin_pp"] = agg["min_relative_return"] - TARGET_RELATIVE_RETURN
agg["drawdown_margin_pp"] = TARGET_MAX_DD - agg["max_max_dd"]
agg["nonzero_cap"] = agg["cap"] > 0.0
agg["simple_signal"] = agg["signal_family"].isin(["short_volshock", "short_drawdown_or_newlow", "short_drawdown_and_vol", "two_of_four_short_risk"])
agg["robust_margin_loss"] = (
    (1.0 - agg["relative_margin_pp"]).clip(lower=0.0) * 10.0
    + (2.0 - agg["drawdown_margin_pp"]).clip(lower=0.0) * 10.0
)
agg = agg.sort_values(["full_gate_all", "robust_margin_loss", "gate_loss", "total_changed_days", "mean_avg_pos", "max_max_dd"], ascending=[False, True, True, True, False, True]).reset_index(drop=True)
agg.to_csv(CARD / "counterfactual_candidate_summary.csv", index=False)

same_day_agg = (
    counterfactual_metrics[counterfactual_metrics["timing"].eq("same_day_diagnostic")]
    .groupby(["candidate", "signal_family", "timing", "cap"], dropna=False)
    .agg(
        fold_count=("fold", "nunique"),
        full_gate_count=("full_gate", "sum"),
        gate_loss=("gate_loss", "sum"),
        max_max_dd=("max_dd", "max"),
        min_relative_return=("relative_return", "min"),
        mean_relative_return=("relative_return", "mean"),
        mean_avg_pos=("avg_pos", "mean"),
        total_changed_days=("changed_days", "sum"),
    )
    .reset_index()
)
same_day_agg["full_gate_all"] = same_day_agg["full_gate_count"].eq(same_day_agg["fold_count"])
same_day_agg = same_day_agg.sort_values(["full_gate_all", "gate_loss", "max_max_dd", "min_relative_return"], ascending=[False, True, True, False]).reset_index(drop=True)
same_day_agg.to_csv(CARD / "same_day_diagnostic_candidate_summary.csv", index=False)

robust_pool = agg[
    agg["full_gate_all"]
    & agg["nonzero_cap"]
    & agg["simple_signal"]
    & (agg["relative_margin_pp"] >= 1.0)
    & (agg["drawdown_margin_pp"] >= 2.0)
].copy()
if robust_pool.empty:
    selection_mode = "best_lagged_gate_loss"
    selected = agg.iloc[0].to_dict()
else:
    selection_mode = "robust_nonzero_cap_with_margin"
    selected = robust_pool.sort_values(["robust_margin_loss", "total_changed_days", "mean_avg_pos", "max_max_dd"], ascending=[True, True, False, True]).iloc[0].to_dict()
selected_rows = primary[
    primary["candidate"].eq(selected["candidate"])
    & primary["signal_family"].eq(selected["signal_family"])
    & primary["timing"].eq(selected["timing"])
    & primary["cap"].eq(selected["cap"])
].sort_values("fold")
selected_rows.to_csv(CARD / "selected_counterfactual_fold_metrics.csv", index=False)

# Persist daily paths for top lagged variants so future notebooks can inspect the path without re-running qlib.
daily_manifest_rows = []
for idx, row in agg.head(5).iterrows():
    for fold in sorted(primary["fold"].unique()):
        key = (str(row["candidate"]), fold, str(row["signal_family"]), str(row["timing"]), float(row["cap"]))
        if key not in daily_outputs:
            continue
        path = DAILY_DIR / f"rank{idx + 1}_{safe_slug(row['candidate'])}_{safe_slug(fold)}_{safe_slug(row['signal_family'])}_{safe_slug(row['timing'])}_cap{float(row['cap']):.2f}.csv"
        daily_outputs[key].to_csv(path, index=False)
        daily_manifest_rows.append({
            "rank": int(idx + 1),
            "candidate": row["candidate"],
            "fold": fold,
            "signal_family": row["signal_family"],
            "timing": row["timing"],
            "cap": float(row["cap"]),
            "path": str(path.relative_to(ROOT)),
        })

daily_path_manifest = pd.DataFrame(daily_manifest_rows)
daily_path_manifest.to_csv(CARD / "counterfactual_daily_manifest.csv", index=False)

display(Markdown("## Top lagged counterfactual variants"))
display(agg.head(20))
display(Markdown("## Selected lagged fold metrics"))
display(selected_rows)
display(Markdown("## Top same-day diagnostics (not eligible for selection)"))
display(same_day_agg.head(10))


## Top lagged counterfactual variants

,candidate,signal_family,timing,cap,fold_count,full_gate_count,gate_loss,max_max_dd,min_relative_return,mean_relative_return,...,total_capped_days,total_signal_days,worst_interval_avg_pos,mean_interval_signal_share,full_gate_all,relative_margin_pp,drawdown_margin_pp,nonzero_cap,simple_signal,robust_margin_loss
0,multi_obj_floor45_cap20_riskfirst,two_of_four_short_risk,lag1,0.00,2,2,0.000000,10.585529,32.364350,57.245824,...,334,352,0.278562,0.368750,True,2.364350,4.414471,False,True,0.000000
1,multi_obj_floor45_cap20_riskfirst,short_volshock,lag1,0.05,2,2,0.000000,11.930788,31.884600,56.257037,...,337,367,0.150103,0.792347,True,1.884600,3.069212,True,True,0.000000
2,multi_obj_floor45_cap20_riskfirst,short_volshock,lag1,0.00,2,2,0.000000,12.218199,30.096972,58.271094,...,349,367,0.120347,0.553922,True,0.096972,2.781801,False,True,9.030275
3,multi_obj_floor45_cap20_riskfirst,short_volshock,lag1,0.08,2,2,0.000000,14.544025,31.295035,53.173110,...,239,367,0.167796,0.792347,True,1.295035,0.455975,True,True,15.440248
4,multi_obj_floor45_cap20_riskfirst,existing_or_two_of_four_short_risk,lag1,0.00,2,2,0.000000,14.688869,30.818276,52.446504,...,351,369,0.040413,0.845831,True,0.818276,0.311131,False,False,18.705927
5,multi_obj_floor45_cap20_riskfirst,two_of_four_short_risk,lag1,0.05,2,2,0.000000,14.968011,34.312948,55.544158,...,322,352,0.168131,0.725236,True,4.312948,0.031989,True,True,19.680112
6,multi_obj_floor45_cap20_riskfirst,existing_or_short_volshock,lag1,0.05,2,1,0.995838,12.176301,29.004162,54.816818,...,348,378,0.146578,0.805167,False,-0.995838,2.823699,True,False,19.958383
7,multi_obj_floor65_cap20_weighted,short_volshock,lag1,0.05,2,1,0.455949,14.139510,29.544051,54.859159,...,354,367,0.152938,0.854618,False,-0.455949,0.860490,True,True,25.954593
8,multi_obj_floor45_cap20_riskfirst,existing_or_short_volshock,lag1,0.08,2,1,0.937089,14.544025,29.062911,52.057047,...,249,378,0.165039,0.805167,False,-0.937089,0.455975,True,False,34.811143
9,multi_obj_floor65_cap20_weighted,short_volshock,lag1,0.00,2,1,2.212729,13.645836,27.787271,56.866243,...,360,367,0.225000,0.606522,False,-2.212729,1.354164,False,True,38.585654


## Selected lagged fold metrics

,candidate,fold,signal_family,timing,cap,source_daily_path,rows,start,end,return,...,capped_days,signal_days,signal_share,position_delta_mean,peak_date,trough_date,interval_days,interval_avg_pos,interval_max_pos,interval_signal_share
29,multi_obj_floor45_cap20_riskfirst,wf_2022,short_volshock,lag1,0.05,runtime/model_selection/card/state_conditioned...,250,2022-01-03,2022-12-29,-5.958598,...,247,249,0.99600,0.114571,2022-01-03,2022-10-14,198,0.050245,0.100000,0.994949
141,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,short_volshock,lag1,0.05,runtime/model_selection/card/state_conditioned...,251,2024-01-02,2024-12-30,31.527965,...,90,118,0.47012,0.029007,2024-03-07,2024-05-01,39,0.150103,0.512402,0.589744


## Top same-day diagnostics (not eligible for selection)

,candidate,signal_family,timing,cap,fold_count,full_gate_count,gate_loss,max_max_dd,min_relative_return,mean_relative_return,mean_avg_pos,total_changed_days,full_gate_all
0,multi_obj_floor45_cap20_riskfirst,short_volshock,same_day_diagnostic,0.00,2,2,0.000000,9.629869,37.145846,61.866959,0.085780,350,True
1,multi_obj_floor45_cap20_riskfirst,short_volshock,same_day_diagnostic,0.05,2,2,0.000000,9.822303,37.071485,58.884114,0.120680,338,True
2,multi_obj_floor45_cap20_riskfirst,existing_or_short_volshock,same_day_diagnostic,0.05,2,2,0.000000,10.098014,32.960563,56.828653,0.118602,349,True
3,multi_obj_floor45_cap20_riskfirst,existing_or_short_volshock,same_day_diagnostic,0.00,2,2,0.000000,11.239766,31.176518,58.882295,0.082607,361,True
4,multi_obj_floor65_cap20_weighted,short_volshock,same_day_diagnostic,0.05,2,2,0.000000,14.164692,37.119116,58.915810,0.128613,355,True
5,multi_obj_floor65_cap20_weighted,short_volshock,same_day_diagnostic,0.00,2,2,0.000000,14.249149,37.192468,61.890270,0.092612,361,True
6,multi_obj_floor45_cap20_riskfirst,short_volshock,same_day_diagnostic,0.08,2,2,0.000000,14.519574,35.245301,55.160898,0.137213,239,True
7,multi_obj_floor45_cap20_riskfirst,existing_or_short_volshock,same_day_diagnostic,0.08,2,2,0.000000,14.519574,32.294029,53.685262,0.135792,250,True
8,multi_obj_floor45_cap20_riskfirst,two_of_four_short_risk,same_day_diagnostic,0.00,2,1,0.694032,15.069403,35.297284,58.922123,0.097008,335,False
9,multi_obj_floor65_cap20_weighted,short_volshock,same_day_diagnostic,0.08,2,1,1.314302,15.131430,35.594807,55.854410,0.146243,246,False


In [5]:

full_gate_count = int(agg["full_gate_all"].sum())
selected_candidate = str(selected["candidate"])
selected_signal = str(selected["signal_family"])
selected_cap = float(selected["cap"])
selected_full_gate = bool(selected["full_gate_all"])
selected_min_relative = float(selected["min_relative_return"])
selected_max_dd = float(selected["max_max_dd"])
selected_gate_loss = float(selected["gate_loss"])
selected_relative_margin = float(selected.get("relative_margin_pp", selected_min_relative - TARGET_RELATIVE_RETURN))
selected_drawdown_margin = float(selected.get("drawdown_margin_pp", TARGET_MAX_DD - selected_max_dd))
source_selected_candidate = str(source_summary["selected_candidate"])
source_selected = source_summary["selected_summary"]
source_selected_metrics = {
    "candidate": source_selected_candidate,
    "wf_2022_relative_return": source_selected["wf_2022_relative_return"],
    "wf_2022_max_dd": source_selected["wf_2022_max_dd"],
    "wf_2023_2024_relative_return": source_selected["wf_2023_2024_relative_return"],
    "wf_2023_2024_max_dd": source_selected["wf_2023_2024_max_dd"],
}

if selected_full_gate:
    decision = "diagnostic_full_gate_counterfactual_needs_official_replay_no_promotion"
    conclusion = [
        f"A lagged counterfactual clears the diagnostic +{TARGET_RELATIVE_RETURN:.0f}pp / <= {TARGET_MAX_DD:.0f}% gate with a robust nonzero-cap preference: {selected_candidate} using {selected_signal} cap={selected_cap:.2f}.",
        f"Selected margins are relative +{selected_relative_margin:.2f}pp above target and drawdown {selected_drawdown_margin:.2f}pp below target; selection_mode={selection_mode}.",
        "This is not a promotion because the cap was applied to saved daily paths outside the official replay/runtime execution layer.",
        "Next action: integrate this lagged short-volatility risk cap into an official warm-start replay or checkpoint-scoring screen and verify transaction costs/action semantics.",
    ]
else:
    decision = "risk_cap_counterfactual_failed_no_promotion"
    conclusion = [
        f"No lagged risk-cap counterfactual clears the full diagnostic gate across both folds; full_gate_all count={full_gate_count}.",
        f"Closest lagged variant is {selected_candidate} / {selected_signal} / cap={selected_cap:.2f}: min relative={selected_min_relative:.2f}pp, max MaxDD={selected_max_dd:.2f}%, summed gate_loss={selected_gate_loss:.2f}, selection_mode={selection_mode}.",
        "The current bottleneck is not solved by a simple post-hoc short-risk cap. The next step should move the short-horizon drawdown/volshock state into checkpoint scoring or training targets, while keeping official replay validation as the gate.",
    ]

selection_payload = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention": "none; fixed-actor daily-path diagnostic only",
    "selection_mode": selection_mode,
    "selected_lagged_variant": json_safe(selected),
    "selected_lagged_fold_metrics": selected_rows.to_dict("records"),
    "source_selected_metrics": source_selected_metrics,
    "full_gate_lagged_variant_count": full_gate_count,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "primary_timing": PRIMARY_TIMING,
    "next_action": "official replay/checkpoint-scoring screen if diagnostic gate passes; otherwise train/checkpoint-score short-horizon risk state rather than post-hoc cap",
}
summary_payload = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": source_summary["canonical_baseline"],
    "direct_comparison_baseline": ["state_conditioned_multi_objective_bc_screen_v1", "drawdown_interval_risk_state_coverage_v1", "sequence_trend_joint_warm_start_official_eval_v1"],
    "profile": PROFILE_NAME,
    "role": "diagnose",
    "changed_layer": ["model_selection", "execution", "risk_state_representation"],
    "external_sources": EXTERNAL_SOURCES,
    "closest_historical_success_case": "drawdown_interval_risk_state_coverage_v1 found short-horizon risk masks covering the 2024 drawdown interval better than long-horizon state",
    "closest_historical_failure_case": "state_conditioned_multi_objective_bc_screen_v1 improved relative return but failed MaxDD after raising reentry participation",
    "duplicate_check": "This run does not rename a profile or tune PPO; it separates actor learning from risk-state execution by applying fold-local, lagged risk masks to saved official daily paths.",
    "source_selected_metrics": source_selected_metrics,
    "selection_mode": selection_mode,
    "selected_lagged_variant": json_safe(selected),
    "selected_lagged_fold_metrics": selected_rows.to_dict("records"),
    "full_gate_lagged_variant_count": full_gate_count,
    "promotion_allowed": False,
    "ppo_screen_allowed_next": False,
    "decision": decision,
    "conclusion": conclusion,
    "sidecars": [
        "signal_test_coverage.csv",
        "baseline_price_replay_metrics.csv",
        "reconstruction_error.csv",
        "counterfactual_metrics.csv",
        "counterfactual_candidate_summary.csv",
        "same_day_diagnostic_candidate_summary.csv",
        "selected_counterfactual_fold_metrics.csv",
        "counterfactual_daily_manifest.csv",
        "daily_paths/*.csv",
        "selection.json",
        "summary.json",
    ],
}

write_json(CARD / "selection.json", selection_payload)
write_json(CARD / "summary.json", summary_payload)

conclusion_md = "\n".join([f"- {item}" for item in conclusion])
selected_table = selected_rows[["fold", "relative_return", "max_dd", "avg_pos", "changed_days", "signal_days", "interval_avg_pos", "interval_signal_share"]].copy()
display(Markdown(f"""
## Conclusion

{conclusion_md}

### Source selected official baseline

- `wf_2022`: relative `{source_selected_metrics['wf_2022_relative_return']:.2f}pp`, MaxDD `{source_selected_metrics['wf_2022_max_dd']:.2f}%`
- `wf_2023_2024`: relative `{source_selected_metrics['wf_2023_2024_relative_return']:.2f}pp`, MaxDD `{source_selected_metrics['wf_2023_2024_max_dd']:.2f}%`

### Selected lagged counterfactual

- candidate: `{selected_candidate}`
- signal: `{selected_signal}`
- cap: `{selected_cap:.2f}`
- selection mode: `{selection_mode}`
- min relative: `{selected_min_relative:.2f}pp`
- max MaxDD: `{selected_max_dd:.2f}%`
- relative margin: `{selected_relative_margin:.2f}pp`
- drawdown margin: `{selected_drawdown_margin:.2f}pp`
- diagnostic full gate: `{selected_full_gate}`
- promotion: `False`
"""))
display(selected_table)



## Conclusion

- A lagged counterfactual clears the diagnostic +30pp / <= 15% gate with a robust nonzero-cap preference: multi_obj_floor45_cap20_riskfirst using short_volshock cap=0.05.
- Selected margins are relative +1.88pp above target and drawdown 3.07pp below target; selection_mode=robust_nonzero_cap_with_margin.
- This is not a promotion because the cap was applied to saved daily paths outside the official replay/runtime execution layer.
- Next action: integrate this lagged short-volatility risk cap into an official warm-start replay or checkpoint-scoring screen and verify transaction costs/action semantics.

### Source selected official baseline

- `wf_2022`: relative `59.39pp`, MaxDD `30.63%`
- `wf_2023_2024`: relative `30.26pp`, MaxDD `16.12%`

### Selected lagged counterfactual

- candidate: `multi_obj_floor45_cap20_riskfirst`
- signal: `short_volshock`
- cap: `0.05`
- selection mode: `robust_nonzero_cap_with_margin`
- min relative: `31.88pp`
- max MaxDD: `11.93%`
- relative margin: `1.88pp`
- drawdown margin: `3.07pp`
- diagnostic full gate: `True`
- promotion: `False`


,fold,relative_return,max_dd,avg_pos,changed_days,signal_days,interval_avg_pos,interval_signal_share
29,wf_2022,80.629474,8.213095,0.050194,247,249,0.050245,0.994949
141,wf_2023_2024,31.884600,11.930788,0.192752,90,118,0.150103,0.589744
